In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv('../data/cleaned_diabetes.csv')

print(f"Cleaned dataset shape: {df.shape}")
print(f"Unique patients: {df['patient_nbr'].nunique():,}")
print(f"Encounters per patient (top 5):")
print(df['patient_nbr'].value_counts().head())

Cleaned dataset shape: (97822, 50)
Unique patients: 68,885
Encounters per patient (top 5):
patient_nbr
88785891    39
1660293     23
23199021    23
88227540    23
23643405    22
Name: count, dtype: int64


In [3]:
# 80/20 split, grouped by patient_nbr
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

train_idx, test_idx = next(splitter.split(df, groups=df['patient_nbr']))
df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

print(f"Train: {len(df_train):,} encounters ({df_train['patient_nbr'].nunique():,} patients)")
print(f"Test:  {len(df_test):,} encounters ({df_test['patient_nbr'].nunique():,} patients)")

# Verify no patient leakage across split
train_patients = set(df_train['patient_nbr'])
test_patients = set(df_test['patient_nbr'])
overlap = train_patients & test_patients
print(f"\nPatient overlap between train and test: {len(overlap)}")
print(f"If 0, the split is leakage-safe.")

# Verify stratification didn't break (post-hoc check)
print(f"\nTrain target rate: {df_train['target'].mean()*100:.2f}%")
print(f"Test target rate:  {df_test['target'].mean()*100:.2f}%")
print(f"Should both be near 11.46%")

Train: 78,283 encounters (55,108 patients)
Test:  19,539 encounters (13,777 patients)

Patient overlap between train and test: 0
If 0, the split is leakage-safe.

Train target rate: 11.33%
Test target rate:  11.96%
Should both be near 11.46%


In [4]:
df_train.to_csv('../data/df_train.csv', index=False)
df_test.to_csv('../data/df_test.csv', index=False)

print(f"Saved df_train.csv ({len(df_train):,} rows)")
print(f"Saved df_test.csv  ({len(df_test):,} rows)")

Saved df_train.csv (78,283 rows)
Saved df_test.csv  (19,539 rows)


In [5]:
df_train_check = pd.read_csv('../data/df_train.csv')
df_test_check = pd.read_csv('../data/df_test.csv')
assert df_train_check.shape == df_train.shape
assert df_test_check.shape == df_test.shape

In [6]:
feature_cols = [ c for c in df_train.columns if c not in ['encounter_id', 'patient_nbr', 'target']]

X_train = df_train[feature_cols].copy()
y_train = df_train['target'].copy()

X_test = df_test[feature_cols].copy()
y_test = df_test['target'].copy()

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

X_train: (78283, 47), y_train: (78283,)
X_test:  (19539, 47),  y_test:  (19539,)


In [8]:
# get all string-type (categorical) columns
categorical_cols = X_train.select_dtypes(include='object').columns

# loop through and inspect each
for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(X_train[col].value_counts(dropna=False).head(10))



=== race ===
race
Caucasian          58490
AfricanAmerican    14831
unknown             1675
Hispanic            1576
Other               1198
Asian                513
Name: count, dtype: int64

=== gender ===
gender
Female    42187
Male      36096
Name: count, dtype: int64

=== age ===
age
[70-80)     20178
[60-70)     17541
[50-60)     13435
[80-90)     13085
[40-50)      7556
[30-40)      2811
[90-100)     2037
[20-30)      1208
[10-20)       374
[0-10)         58
Name: count, dtype: int64

=== payer_code ===
payer_code
unknown    30971
MC         24770
HM          4914
SP          3810
BC          3627
MD          2750
CP          1926
UN          1884
CM          1522
OG           811
Name: count, dtype: int64

=== medical_specialty ===
medical_specialty
unknown                       38485
InternalMedicine              11286
Emergency/Trauma               5903
Family/GeneralPractice         5652
Cardiology                     4157
Surgery-General                2379
Nephrology   

/tmp/ipykernel_409977/633098692.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include='object').columns


In [9]:
medication_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 
                   'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 
                   'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 
                   'miglitol', 'troglitazone', 'tolazamide', 'examide', 
                   'citoglipton', 'insulin', 'glyburide-metformin', 
                   'glipizide-metformin', 'glimepiride-pioglitazone',
                   'metformin-rosiglitazone', 'metformin-pioglitazone']

X_train[medication_cols].nunique().sort_values()

examide                     1
citoglipton                 1
tolbutamide                 2
acetohexamide               2
glimepiride-pioglitazone    2
metformin-rosiglitazone     2
glipizide-metformin         2
troglitazone                2
metformin-pioglitazone      2
tolazamide                  3
glimepiride                 4
glipizide                   4
glyburide                   4
metformin                   4
nateglinide                 4
repaglinide                 4
miglitol                    4
pioglitazone                4
rosiglitazone               4
acarbose                    4
chlorpropamide              4
insulin                     4
glyburide-metformin         4
dtype: int64

In [10]:
near_constant = ['tolbutamide', 'acetohexamide', 'troglitazone',
                 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
                 'glipizide-metformin', 'metformin-pioglitazone', 'tolazamide']

for col in near_constant:
    print(f"\n{col}:")
    print(X_train[col].value_counts(normalize=True).head())


tolbutamide:
tolbutamide
No        0.999796
Steady    0.000204
Name: proportion, dtype: float64

acetohexamide:
acetohexamide
No        0.999987
Steady    0.000013
Name: proportion, dtype: float64

troglitazone:
troglitazone
No        0.999974
Steady    0.000026
Name: proportion, dtype: float64

glimepiride-pioglitazone:
glimepiride-pioglitazone
No        0.999987
Steady    0.000013
Name: proportion, dtype: float64

metformin-rosiglitazone:
metformin-rosiglitazone
No        0.999987
Steady    0.000013
Name: proportion, dtype: float64

glipizide-metformin:
glipizide-metformin
No        0.999898
Steady    0.000102
Name: proportion, dtype: float64

metformin-pioglitazone:
metformin-pioglitazone
No        0.999987
Steady    0.000013
Name: proportion, dtype: float64

tolazamide:
tolazamide
No        0.999591
Steady    0.000396
Up        0.000013
Name: proportion, dtype: float64


## Low-Variance Feature Pruning

Audit of medication columns revealed 10 features with minority class 
below 1% — far below the threshold required for a model to learn 
reliably from them.

- 2 columns are constants (zero variance) — `examide`, `citoglipton`
- 8 columns are near-constants — minority class ranges from 0.001% to 0.04%

Dropping these reduces feature space without information loss and 
avoids overfitting to clinically rare events. The 13 remaining 
medication columns retain real variation (4 categories with 
substantive support) and represent meaningful clinical decisions.

| Type | Columns | Encoding strategy |
|---|---|---|
| **Already numeric (use as-is)** | time_in_hospital, num_lab_procedures, num_procedures, num_medications, number_outpatient, number_emergency, number_inpatient, number_diagnoses | No transformation needed |
| **Already binary (use as-is)** | weight_recorded, target | No transformation needed |
| **Binary categoricals (map to 0/1)** | gender, change, diabetesMed | Simple mapping |
| **Ordinal (numeric mapping)** | age | Map buckets to integer positions |
| **Multi-class nominal (one-hot)** | race, max_glu_serum, A1Cresult, medical_specialty, payer_code, admission_type_id, discharge_disposition_id, admission_source_id | One-hot encode |
| **Multi-class medications (one-hot or drop)** | All 23 medication columns | Drop constants first, then one-hot the rest |
| **High-cardinality ICD codes** | diag_1, diag_2, diag_3 | Special handling — group or target-encode |

In [12]:
df_train_check = pd.read_csv('../data/df_train.csv')
print(f"Columns in saved file: {df_train_check.shape[1]}")
print(f"Has 'examide'? {'examide' in df_train_check.columns}")

Columns in saved file: 50
Has 'examide'? True


In [14]:

X_train = pd.read_csv('../data/df_train.csv').drop(columns=['encounter_id', 'patient_nbr', 'target'])
y_train = pd.read_csv('../data/df_train.csv')['target']
X_test = pd.read_csv('../data/df_test.csv').drop(columns=['encounter_id', 'patient_nbr', 'target'])
y_test = pd.read_csv('../data/df_test.csv')['target']

# Apply the same low-variance drops
low_variance_features = [
    'examide', 'citoglipton',
    'tolbutamide', 'acetohexamide', 'troglitazone',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'glipizide-metformin', 'metformin-pioglitazone', 'tolazamide'
]
X_train = X_train.drop(columns=low_variance_features)
X_test = X_test.drop(columns=low_variance_features)

# Check cardinality of all categorical columns
cat_cols = X_train.select_dtypes(include=['object', 'string']).columns
print("Unique values per categorical column:")
print(X_train[cat_cols].nunique().sort_values(ascending=False))

Unique values per categorical column:
diag_3                 761
diag_2                 720
diag_1                 689
medical_specialty       72
payer_code              18
age                     10
race                     6
max_glu_serum            4
A1Cresult                4
pioglitazone             4
metformin                4
repaglinide              4
nateglinide              4
chlorpropamide           4
glimepiride              4
glipizide                4
glyburide                4
insulin                  4
rosiglitazone            4
acarbose                 4
miglitol                 4
glyburide-metformin      4
gender                   2
change                   2
diabetesMed              2
dtype: int64


In [15]:
def group_icd9(code):
    """Group ICD-9 codes into 9 clinical categories for readmission analysis.
    
      Based on the standard  groupings used in the readmission ML literature
      for diabetes 130 dataset.
    """

    # handle missing non-numeric codes
    if pd.isna(code):
        return 'Other'
    
    # ICD-9 codes starting with V or E are special (V = supplementary, E = external causes)
    code_str = str(code)
    if code_str.startswith('V') or code_str.startswith('E'):
        return 'Other'
    
    try:
        code_num = float(code_str)
    except ValueError:
        return 'Other'
    
      # Diabetes-specific (this is a diabetes-focused dataset, so we isolate it)
    if 250 <= code_num < 251:
        return 'Diabetes'
    
    # Circulatory (390-459)
    if 390 <= code_num < 460:
        return 'Circulatory'
    
    # Respiratory (460-519)
    if 460 <= code_num < 520:
        return 'Respiratory'
    
    # Digestive (520-579)
    if 520 <= code_num < 580:
        return 'Digestive'
    
    # Genitourinary (580-629)
    if 580 <= code_num < 630:
        return 'Genitourinary'
    
    # Neoplasms (140-239)
    if 140 <= code_num < 240:
        return 'Neoplasms'
    
    # Injury (800-999)
    if 800 <= code_num < 1000:
        return 'Injury'
    
    # Musculoskeletal (710-739)
    if 710 <= code_num < 740:
        return 'Musculoskeletal'
    
    return 'Other'

In [16]:
# Apply the grouping to all three diagnosis columns in BOTH splits
for col in ['diag_1', 'diag_2', 'diag_3']:
    X_train[col] = X_train[col].apply(group_icd9)
    X_test[col] = X_test[col].apply(group_icd9)

# Verify
print("Diag_1 distribution after grouping:")
print(X_train['diag_1'].value_counts())
print(f"\nUnique values per diag column now:")
for col in ['diag_1', 'diag_2', 'diag_3']:
    print(f"  {col}: {X_train[col].nunique()} unique values")

Diag_1 distribution after grouping:
diag_1
Circulatory        23546
Other              17504
Respiratory         7925
Digestive           7180
Diabetes            6415
Injury              5347
Genitourinary       3972
Musculoskeletal     3916
Neoplasms           2478
Name: count, dtype: int64

Unique values per diag column now:
  diag_1: 9 unique values
  diag_2: 9 unique values
  diag_3: 9 unique values


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Numeric features — pass through with scaling
numeric_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses', 'weight_recorded'
]

# Binary categoricals — map to 0/1 via OrdinalEncoder
binary_cols = ['gender', 'change', 'diabetesMed']

# Ordinal — preserve order
ordinal_cols = ['age']
age_order = [['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
              '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']]

# Multi-class nominal — one-hot encode (now includes the grouped diag columns)
nominal_cols = [
    'race', 'max_glu_serum', 'A1Cresult',
    'medical_specialty', 'payer_code',
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'diag_1', 'diag_2', 'diag_3',
    # 13 medications with real variation
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'insulin', 'glyburide-metformin'
]

# Verify total accounting
all_grouped = numeric_cols + binary_cols + ordinal_cols + nominal_cols
missing = set(X_train.columns) - set(all_grouped)
extra = set(all_grouped) - set(X_train.columns)
print(f"Total columns: {X_train.shape[1]}")
print(f"Accounted for: {len(all_grouped)}")
print(f"Missing: {missing}")
print(f"Extra: {extra}")

Total columns: 37
Accounted for: 37
Missing: set()
Extra: set()


In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('bin', OrdinalEncoder(), binary_cols),
        ('ord', OrdinalEncoder(categories=age_order), ordinal_cols),
        ('nom', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), nominal_cols)
    ],
    remainder='drop',
    verbose_feature_names_out=True,
)

# fit on trainning data, transform both splits
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print(f"X_train encoded shape: {X_train_encoded.shape}")
print(f"X_test encoded shape:  {X_test_encoded.shape}")

X_train encoded shape: (78283, 217)
X_test encoded shape:  (19539, 217)


/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [3, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [19]:
# Decode column indices to column names
nominal_transformer = preprocessor.named_transformers_['nom']
nom_cols_in_order = nominal_cols

print("Nominal column indices (from the warning):")
print(f"  Column 3: {nom_cols_in_order[3]}")
print(f"  Column 6: {nom_cols_in_order[6]}")

# For each, check what's in test that wasn't in train
for col_idx in [3, 6]:
    col_name = nom_cols_in_order[col_idx]
    train_categories = set(X_train[col_name].unique())
    test_categories = set(X_test[col_name].unique())
    unknown = test_categories - train_categories
    
    # How many test rows are affected
    affected_rows = X_test[col_name].isin(unknown).sum()
    
    print(f"\n{col_name}:")
    print(f"  Unknown categories in test: {unknown}")
    print(f"  Affected rows: {affected_rows} ({affected_rows / len(X_test) * 100:.2f}% of test)")

Nominal column indices (from the warning):
  Column 3: medical_specialty
  Column 6: discharge_disposition_id

medical_specialty:
  Unknown categories in test: {'Dermatology'}
  Affected rows: 1 (0.01% of test)

discharge_disposition_id:
  Unknown categories in test: {np.int64(12)}
  Affected rows: 3 (0.02% of test)


## Encoder Warning — Investigation

During `transform(X_test)`, the OneHotEncoder reported unknown categories in 
two columns:

- `medical_specialty`: 'Dermatology' (1 test row)
- `discharge_disposition_id`: 12 (3 test rows)

Total: 4 rows / 19,539 (0.02% of test set).

This is the expected consequence of patient-grouped train/test splitting — 
rare categories occasionally land entirely on one side. `handle_unknown='ignore'` 
correctly handles these as all-zeros in the affected encoded columns. The 
impact on test performance is negligible (4/19,539 rows partially mis-encoded), 
and the alternative (raising an error) would prevent deployment to any new 
data containing previously unseen categories.

**Decision: accept and proceed.**